In [6]:
# Cell 1: Install required packages
# !pip install gradio groq torch transformers pillow matplotlib requests
!pip install groq

# Cell 2: Import libraries
import torch
import gc
import gradio as gr
from transformers import pipeline
from PIL import Image
import requests
import matplotlib.pyplot as plt
from io import BytesIO
import numpy as np
from groq import Groq
import os


In [2]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
# Cell 3: Setup Groq API (you'll need to add your API key)

from groq import Groq
GROQ_API_KEY = ""  # Replace with your actual API key
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Cell 4: Initialize Groq client
def init_groq():
    """Initialize Groq client with API key"""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key or api_key == "your-groq-api-key-here":
        print("⚠️ Please set your Groq API key in Cell 3")
        return None
    return Groq(api_key=api_key)

groq_client = init_groq()

In [4]:
# In a new cell, run this first
import torch
import gc

# Clear all cached memory
torch.cuda.empty_cache()
gc.collect()

# Reset the CUDA device (this is more aggressive)
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()

# Check current memory
print(f"Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

# If still high, restart runtime (Runtime → Restart runtime)

Allocated: 0.00 GB
Reserved: 0.00 GB


In [8]:
# Cell 5: Load MedGemma with proper optimizations
import torch
import gc
from transformers import pipeline
from transformers import BitsAndBytesConfig
import torch

print("Loading MedGemma with speed optimizations...")
torch.cuda.empty_cache()
gc.collect()

# Option 1: Standard loading (most compatible)
print("Loading MedGemma (this takes 1-2 minutes)...")
pipe = pipeline(
    "image-text-to-text",
    model="google/medgemma-1.5-4b-it",
    token = '', #paste a token here
    torch_dtype=torch.bfloat16,  # Use bfloat16 for speed
    device="cuda" if torch.cuda.is_available() else "cpu",
    model_kwargs={
        "low_cpu_mem_usage": True,
        "offload_folder": "offload",
    }
)

# Check memory usage
if torch.cuda.is_available():
    print(f"✓ MedGemma loaded. GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
else:
    print("✓ MedGemma loaded on CPU")

Loading MedGemma with speed optimizations...
Loading MedGemma (this takes 1-2 minutes)...


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✓ MedGemma loaded. GPU memory: 8.02 GB


In [9]:
# Cell 6: Define formatting prompt for Groq - FOCUSED ON PNEUMONIA
def get_formatting_prompt(medgemma_output):
    """
    Create a prompt for Groq to format the medical analysis with focus on pneumonia detection
    """
    return f"""You are a specialist in chest X-ray analysis with expertise in pneumonia detection. Please convert the following X-ray analysis into a structured medical report with clear emphasis on pneumonia findings.

Original MedGemma Analysis:
{medgemma_output}

Please format this into a professional medical report with the following structure:

---
**RADIOLOGY REPORT - PNEUMONIA ASSESSMENT**

**EXAMINATION:** Chest X-Ray

**PNEUMONIA STATUS:** [POSITIVE / NEGATIVE / INDETERMINATE]

**CONFIDENCE LEVEL:** [High / Medium / Low]

**KEY FINDINGS FOR PNEUMONIA:**
- [Presence/absence of consolidation]
- [Presence/absence of infiltrates]
- [Air bronchograms if present]
- [Lobar vs multifocal involvement]
- [Unilateral vs bilateral findings]

**OTHER FINDINGS:**
- [Any other abnormalities not related to pneumonia]

**IMPRESSION:**
- [Clear statement: Pneumonia present / No evidence of pneumonia / Indeterminate - correlation needed]

**RECOMMENDATIONS:**
- [Clinical correlation advised if positive]
- [Follow-up imaging if needed]
- [Additional tests if indicated]

**NOTE:** This is an AI-assisted screening tool for pneumonia detection and should be reviewed by a qualified healthcare professional.
---

Make it:
- Focus primarily on pneumonia-related findings
- Clearly indicate if pneumonia is present or absent
- Use standardized terminology for pneumonia (consolidation, infiltrates, air bronchograms)
- Include confidence level in the assessment
- Structured for quick clinical decision-making
"""

In [10]:
# Cell 7: Optimized MedGemma analysis function - FOCUSED ON PNEUMONIA
def analyze_xray_pipeline(image):
    """
    Optimized MedGemma analysis with focus on pneumonia detection
    """
    results = {"medgemma_raw": "", "formatted_report": "", "status": "success", "pneumonia_detected": "unknown"}

    try:
        if image is None:
            return {"status": "error", "medgemma_raw": "No image", "formatted_report": "Please upload an image"}

        # SPEED BOOST 1: Resize image to smaller size
        original_size = image.size
        max_size = 512
        image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        print(f"  ✓ Image resized from {original_size} to {image.size}")

        # Convert to grayscale for X-ray
        if image.mode != 'L':
            image = image.convert('L')

        # SPEED BOOST 2: Use PNEUMONIA-FOCUSED prompt
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": """Analyze this chest X-ray specifically for pneumonia. Answer these questions:
                1. Is pneumonia present? (Yes/No/Uncertain)
                2. What specific findings support this? (consolidation, infiltrates, air bronchograms)
                3. Which lung regions are affected? (upper/lower, left/right, unilateral/bilateral)
                4. How confident are you in this assessment? (High/Medium/Low)
                5. Are there any other findings not related to pneumonia?"""}
            ]
        }]

        # Clear cache before inference
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        # Generate response from MedGemma
        with torch.no_grad():
            output = pipe(
                text=messages,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=1
            )

        medgemma_response = output[0]["generated_text"][-1]["content"]
        results["medgemma_raw"] = medgemma_response

        # Quick pneumonia detection from raw response (simple keyword check)
        pneumonia_keywords = ['pneumonia', 'consolidation', 'infiltrate', 'air bronchogram',
                             'opacity', 'lobar', 'multifocal']
        medgemma_lower = medgemma_response.lower()

        if any(keyword in medgemma_lower for keyword in pneumonia_keywords):
            if 'no pneumonia' in medgemma_lower or 'without pneumonia' in medgemma_lower or 'negative' in medgemma_lower:
                results["pneumonia_detected"] = "negative"
            else:
                results["pneumonia_detected"] = "positive"
        else:
            results["pneumonia_detected"] = "negative"

        # Format with Groq using a CURRENTLY SUPPORTED model
        if groq_client:
            try:
                # Using llama-3.1-8b-instant which is currently supported
                chat_completion = groq_client.chat.completions.create(
                    messages=[
                        {"role": "system", "content": "You are a pneumonia detection specialist. Format this X-ray finding with focus on pneumonia assessment."},
                        {"role": "user", "content": medgemma_response}
                    ],
                    model="llama-3.1-8b-instant",  # Updated to supported model
                    temperature=0.1,
                    max_tokens=400
                )
                results["formatted_report"] = chat_completion.choices[0].message.content
            except Exception as groq_error:
                print(f"⚠️ Groq formatting failed: {groq_error}")
                # Fallback to MedGemma response if Groq fails
                results["formatted_report"] = f"""PNEUMONIA ASSESSMENT (Direct from MedGemma):

{medgemma_response}

---
Note: This is unformatted output. Groq formatting service temporarily unavailable."""
        else:
            results["formatted_report"] = medgemma_response

        return results

    except Exception as e:
        print(f"Error in analysis: {e}")
        return {"status": "error", "medgemma_raw": str(e),
                "formatted_report": f"Analysis failed: {str(e)}",
                "pneumonia_detected": "error"}

In [11]:
# Cell 8: Upload and analyze with PNEUMONIA FOCUS
from google.colab import files
from PIL import Image
import io
import time

print("="*60)
print("🏥 PNEUMONIA DETECTION SYSTEM")
print("="*60)
print("\n📤 Upload a chest X-ray to check for pneumonia")
print("⏱️  Analysis takes 2-3 minutes")
print("📋 This system specifically detects pneumonia vs normal findings\n")

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*60}")
    print(f"📂 Processing: {filename}")
    print(f"{'='*60}")

    start_time = time.time()

    # Load image
    print("⏳ Loading X-ray image...")
    image = Image.open(io.BytesIO(uploaded[filename]))

    # Analyze with pneumonia focus
    print("⏳ MedGemma analyzing for pneumonia...")
    results = analyze_xray_pipeline(image)

    elapsed = time.time() - start_time

    # Show pneumonia status prominently
    print(f"\n✅ Analysis complete in {elapsed:.1f} seconds")
    print("\n" + "★"*60)

    # Pneumonia status with color coding in text
    pneumonia_status = results.get('pneumonia_detected', 'unknown').upper()
    if pneumonia_status == 'POSITIVE':
        status_display = "⚠️ PNEUMONIA DETECTED ⚠️"
    elif pneumonia_status == 'NEGATIVE':
        status_display = "✅ NO PNEUMONIA DETECTED"
    else:
        status_display = "❓ PNEUMONIA STATUS UNCERTAIN"

    print(f"📋 PNEUMONIA ASSESSMENT: {status_display}")
    print("★"*60)

    # Show formatted report
    print("\n📋 DETAILED RADIOLOGY REPORT")
    print("-"*60)
    print(results["formatted_report"])
    print("-"*60)

    # Optional: Show raw analysis for reference
    print("\n" + "☆"*60)
    print("🔬 RAW PNEUMONIA ANALYSIS")
    print("☆"*60)
    raw_preview = results["medgemma_raw"][:300] + "..." if len(results["medgemma_raw"]) > 300 else results["medgemma_raw"]
    print(raw_preview)

    # Add clinical recommendation based on pneumonia status
    print("\n" + "⚕️"*30)
    if pneumonia_status == 'POSITIVE':
        print("⚕️ CLINICAL RECOMMENDATION: Patient should be evaluated for pneumonia treatment.")
        print("⚕️ Consider: antibiotics, further imaging, or hospitalization if severe.")
    elif pneumonia_status == 'NEGATIVE':
        print("⚕️ CLINICAL RECOMMENDATION: No immediate pneumonia concerns. Follow up if symptoms persist.")
    else:
        print("⚕️ CLINICAL RECOMMENDATION: Additional clinical correlation and imaging may be needed.")
    print("⚕️"*30)

    print(f"\n{'='*60}")
    print(f"📊 SUMMARY: {status_display}")
    print(f"⏱️  Total time: {elapsed:.1f} seconds")
    print(f"{'='*60}")

    # Cleanup
    del image
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Optional: Ask if user wants to analyze another image
    print("\n" + "-"*60)
    another = input("Analyze another X-ray? (y/n): ").strip().lower()
    if another != 'y':
        print("\n👋 Thank you for using Pneumonia Detection System!")
        break

🏥 PNEUMONIA DETECTION SYSTEM

📤 Upload a chest X-ray to check for pneumonia
⏱️  Analysis takes 2-3 minutes
📋 This system specifically detects pneumonia vs normal findings



Saving pneumonia1.jpg to pneumonia1.jpg

📂 Processing: pneumonia1.jpg
⏳ Loading X-ray image...
⏳ MedGemma analyzing for pneumonia...
  ✓ Image resized from (356, 357) to (356, 357)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Analysis complete in 20.5 seconds

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
📋 PNEUMONIA ASSESSMENT: ⚠️ PNEUMONIA DETECTED ⚠️
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📋 DETAILED RADIOLOGY REPORT
------------------------------------------------------------
**Pneumonia Detection Assessment**

**Patient Information:** (Not provided)

**X-ray Image Quality:** Blurry

**Pneumonia Assessment:**

1. **Is pneumonia present?** Uncertain
2. **What specific findings support this?** 
   - No clear consolidation
   - No clear infiltrates
   - No air bronchograms
   - Insufficient image quality to make a definitive assessment
3. **Which lung regions are affected?** Not enough information to determine the affected lung regions
4. **How confident are you in this assessment?** Low
5. **Are there any other findings not related to pneumonia?** Insufficient image quality to identify any other findings

**Recommendations:**

- Obtain a high-quality chest X-ray ima

In [14]:
# Cell 9: check some images misclassified by CNN
from google.colab import files
from PIL import Image
import io
import time

print("="*60)
print("🏥 PNEUMONIA DETECTION SYSTEM")
print("="*60)
print("\n📤 Upload a chest X-ray to check for pneumonia")
print("⏱️  Analysis takes 2-3 minutes")
print("📋 This system specifically detects pneumonia vs normal findings\n")

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*60}")
    print(f"📂 Processing: {filename}")
    print(f"{'='*60}")

    start_time = time.time()

    # Load image
    print("⏳ Loading X-ray image...")
    image = Image.open(io.BytesIO(uploaded[filename]))

    # Analyze with pneumonia focus
    print("⏳ MedGemma analyzing for pneumonia...")
    results = analyze_xray_pipeline(image)

    elapsed = time.time() - start_time

    # Show pneumonia status prominently
    print(f"\n✅ Analysis complete in {elapsed:.1f} seconds")
    print("\n" + "★"*60)

    # Pneumonia status with color coding in text
    pneumonia_status = results.get('pneumonia_detected', 'unknown').upper()
    if pneumonia_status == 'POSITIVE':
        status_display = "⚠️ PNEUMONIA DETECTED ⚠️"
    elif pneumonia_status == 'NEGATIVE':
        status_display = "✅ NO PNEUMONIA DETECTED"
    else:
        status_display = "❓ PNEUMONIA STATUS UNCERTAIN"

    print(f"📋 PNEUMONIA ASSESSMENT: {status_display}")
    print("★"*60)

    # Show formatted report
    print("\n📋 DETAILED RADIOLOGY REPORT")
    print("-"*60)
    print(results["formatted_report"])
    print("-"*60)

    # Optional: Show raw analysis for reference
    print("\n" + "☆"*60)
    print("🔬 RAW PNEUMONIA ANALYSIS")
    print("☆"*60)
    raw_preview = results["medgemma_raw"][:300] + "..." if len(results["medgemma_raw"]) > 300 else results["medgemma_raw"]
    print(raw_preview)

    # Add clinical recommendation based on pneumonia status
    print("\n" + "⚕️"*30)
    if pneumonia_status == 'POSITIVE':
        print("⚕️ CLINICAL RECOMMENDATION: Patient should be evaluated for pneumonia treatment.")
        print("⚕️ Consider: antibiotics, further imaging, or hospitalization if severe.")
    elif pneumonia_status == 'NEGATIVE':
        print("⚕️ CLINICAL RECOMMENDATION: No immediate pneumonia concerns. Follow up if symptoms persist.")
    else:
        print("⚕️ CLINICAL RECOMMENDATION: Additional clinical correlation and imaging may be needed.")
    print("⚕️"*30)

    print(f"\n{'='*60}")
    print(f"📊 SUMMARY: {status_display}")
    print(f"⏱️  Total time: {elapsed:.1f} seconds")
    print(f"{'='*60}")

    # Cleanup
    del image
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    # Optional: Ask if user wants to analyze another image
    print("\n" + "-"*60)
    another = input("Analyze another X-ray? (y/n): ").strip().lower()
    if another != 'y':
        print("\n👋 Thank you for using Pneumonia Detection System!")
        break

🏥 PNEUMONIA DETECTION SYSTEM

📤 Upload a chest X-ray to check for pneumonia
⏱️  Analysis takes 2-3 minutes
📋 This system specifically detects pneumonia vs normal findings



Saving misclassifiedbyCNN_Normal_Pred_Pneumonia2.jpg to misclassifiedbyCNN_Normal_Pred_Pneumonia2.jpg

📂 Processing: misclassifiedbyCNN_Normal_Pred_Pneumonia2.jpg
⏳ Loading X-ray image...
⏳ MedGemma analyzing for pneumonia...
  ✓ Image resized from (350, 356) to (350, 356)


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✅ Analysis complete in 18.5 seconds

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
📋 PNEUMONIA ASSESSMENT: ⚠️ PNEUMONIA DETECTED ⚠️
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📋 DETAILED RADIOLOGY REPORT
------------------------------------------------------------
**Pneumonia Detection Specialist's Report**

**Case Number:** [Insert Case Number]

**Patient Information:** [Insert Patient Information]

**Chest X-ray Analysis:**

**1. Pneumonia Presence:** Uncertain

**Reasoning:** The provided chest X-ray image is too blurry to confidently assess for pneumonia. The characteristic features of pneumonia, such as consolidation, infiltrates, or air bronchograms, are not evident in this image.

**2. Specific Findings Supporting this Assessment:**

* No clear consolidation or infiltrates are visible.
* No air bronchograms are present.
* The image quality is poor, making it difficult to assess for other findings.

**3. Affected Lung Regions:** Unable to Determi